# Guitar Chord Lab 3：単独ギター・混合音・分離音の比較実験

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/guitar-chord-lab/notebooks/03_comparison_experiment.ipynb)　[GitHubで見る](https://github.com/shogo-ishikawa/2026_3Q_M4Th4/blob/main/projects/guitar-chord-lab/notebooks/03_comparison_experiment.ipynb)

研究の問い：**混合音源からギターのコードを推定するとき、ギターを分離する前処理はどのような伴奏の条件で役立つか。**

同じ認識器・同じ設定で、次の3条件を比べます。

| 条件 | 内容 | 役割 |
| --- | --- | --- |
| solo | 正解注釈の付いた単独ギター | 伴奏がない場合の比較基準（正解ラベルとしては使わない） |
| mixture | ギターに伴奏を混ぜた音 | 分離しない場合の結果 |
| separated | mixture から Demucs で取り出したギター | 分離による変化を調べる対象 |

- 所要時間の目安：この教材を確認した環境（CPU 4コア、GPU なし）では、データの取得を含めて約4〜5分でした（5録音 × 伴奏3段階 = 15回の分離）。Colab の CPU ランタイムでは、より長くかかる可能性があります。
- 新規性や精度向上を前提にしない実験です。分離で悪化する条件が見つかっても、そのまま記録します。

## 1. セットアップ

新しいColabでは、講義リポジトリをHTTPSで取得し、`projects/guitar-chord-lab/` の作品だけをインストールします。

| 項目 | 場所 |
| --- | --- |
| リポジトリの取得先 | `/content/2026_3Q_M4Th4` |
| 作業ディレクトリ（相対パスの基準） | `/content/2026_3Q_M4Th4/projects/guitar-chord-lab` |
| 結果の保存先 | `outputs/experiment_guitarset/` または `outputs/experiment_demo/`（作業ディレクトリからの相対パス） |

- 表示されるコミットは、使ったコードの版です。結果と一緒に記録してください。
- `BRANCH`（ブランチ）や `COMMIT`（コミット）を変えて実行し直すと、その版にそろえます。コードの版が変わったときは、セッションの再起動を求める表示が出ます。
- Colabのファイルはランタイムを終了すると消えます。残したい結果はダウンロードするか、Googleドライブにコピーしてください。
- ローカルで使う場合は、作品フォルダで `pip install -e ".[separation]"` を実行してから、`notebooks/` フォルダでこのNotebookを開きます。


In [ ]:
# セットアップ（最初に実行します。BRANCH や COMMIT を変えたときも実行し直します）
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
BRANCH = "main"  # 作業ブランチの内容を試すときは、ブランチ名に書き換えます
COMMIT = ""      # 実験を再現するときは、記録したコミット（例: "1a2b3c4"）を入れます
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/2026_3Q_M4Th4")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    # 実行するたびに BRANCH（COMMIT があればそのコミット）の内容にそろえます
    git = ["git", "-C", str(REPO_DIR)]
    before = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True).stdout
    subprocess.run(git + ["fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(git + ["checkout", "-q", "--detach", COMMIT or "FETCH_HEAD"], check=True)
    after = subprocess.run(git + ["rev-parse", "HEAD"], capture_output=True, text=True).stdout
    if before != after and "guitar_chord_lab" in sys.modules:
        raise RuntimeError("コードの版が変わりました。「ランタイム」→「セッションを再起動」を選んでから、このセルから実行し直してください")
    PROJECT_DIR = REPO_DIR / "projects" / "guitar-chord-lab"
    # この作品だけをインストールします（[separation]）
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{PROJECT_DIR}[separation]"], check=True)
else:
    # ローカルでは、notebooks/ フォルダで開いたこのNotebookの1つ上を作品フォルダとします
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)  # 以降の相対パスは、作品フォルダ（projects/guitar-chord-lab）が基準です
sys.path.insert(0, str(PROJECT_DIR / "src"))  # インストール直後でも import できるようにします
commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("作業ディレクトリ:", Path.cwd())
print("使用しているコミット:", commit or "（Gitの管理外。配布ZIPなら release_info.json を参照）")


In [ ]:
%matplotlib inline
import csv

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, Image, display

import guitar_chord_lab as gcl
from guitar_chord_lab.experiment import (
    check_source_split, count_materials, load_experiment, load_listening_clips, run_experiment,
)
from guitar_chord_lab.plots import plot_analysis, plot_comparison
from guitar_chord_lab.recognition import recognise_chords

print("Guitar Chord Lab", gcl.__version__)

## 2. 実験の決まりごと

**評価の指標**（`src/guitar_chord_lab/evaluation.py`）
- 主指標：正解率 ＝ 正解コードと予測が一致した時間 ÷ 評価対象の時間。正解と予測の**区間が重なる時間**から計算します（フレーム番号では対応付けません）。
- 判定保留と予測のない時間は、評価対象から除かず**不正解**として数えます。判定できた時間の割合（判定率）も別に表示します。
- 正解注釈のない時間は評価しません（N と同じ意味にはしません）。

**比べ方**
- 同じ録音・同じ伴奏・同じ音量どうしで「分離後 − 分離前」の差を計算します。
- 全体の平均だけでなく、録音ごとの値・ばらつき（標準偏差、最小、最大）を確認します。
- **元録音数と混合条件数を分けて数えます**。例えば5録音に3段階の音量を使ったら「5素材・15混合条件」です。同じ録音の加工物を独立した曲として数えません。

**調整用と評価用の分割**
- 設定（時間幅やしきい値）を選ぶときは調整用（tuning）の録音だけを使い、最終的な評価は評価用（evaluation）の録音で行います。
- 分割は**元素材単位**です。同じ録音の切り出し・音量違い・分離結果を両側に入れません。
- この作品の既定の設定は、GuitarSet の奏者00の録音（調整用）だけを見て選びました。このNotebookの既定の素材は奏者01〜05の録音（評価用）です。
- ただし、例の録音を選ぶ規則（下の3.）は、これらの録音を含む120本の単独ギターの推定結果を一度見た後で決めました。規則は注釈だけを使いますが、厳密には「一度も見ていない評価用データ」ではありません（`docs/research-guide.md` の 8.）。

**素材の区別**
- 人工的に混ぜた音（artificial）と、実際に合奏を録音した音（recorded）を区別して記録します。
- 合成音（synthetic）と実際の録音（real）も区別します。合成音の結果を実ギターの性能の根拠にしません。

## 3. 素材を準備する

| MATERIAL | 内容 |
| --- | --- |
| `"guitarset"`（既定） | GuitarSet の実ギター録音5本（奏者01〜05、各20秒）＋合成ドラム。約15 MB をダウンロード |
| `"demo"` | 合成デモ2本＋合成ドラム。プログラムの動作確認用（分離の評価には使えません） |

GuitarSet の正解は、リードシートの「指示されたコード」です。ギターが3度の音（メジャーとマイナーを分ける音）をほとんど弾いていない録音では、ギターの音だけからは正解を判断できません。そこで、音符の注釈から「3度の音が少なくとも一度弾かれたコード区間の、時間の割合」（`third_played_ratio`）を計算して記録し、例には 0.9 以上の録音を選んでいます。3度の音が鳴り続けている時間の割合ではありません。選定の規則は注釈だけを使いますが、規則を決める前に推定結果を一度見ています（2.の注意を参照）。

In [ ]:
MATERIAL = "guitarset"          # "guitarset"（実ギター）または "demo"（合成音、動作確認用）
LEVELS_DB = [0.0, 6.0, 12.0]    # 伴奏の音量（伴奏のRMS − ギターのRMS）[dB]

if MATERIAL == "guitarset":
    from guitar_chord_lab.guitarset import CITATION, EXAMPLE_TRACKS, prepare_guitarset_experiment

    print("出典:", CITATION)
    config_path = prepare_guitarset_experiment(
        EXAMPLE_TRACKS, "data/raw/guitarset", "data/raw/guitarset/experiment_guitarset.toml",
        duration_s=20.0, levels_db=LEVELS_DB,
    )
else:
    from guitar_chord_lab.synth import write_demo_dataset

    write_demo_dataset("data/generated/demo")
    config_path = "configs/experiment_demo.toml"

experiment = load_experiment(config_path)
experiment.accompaniment_db = LEVELS_DB
print("設定ファイル:", config_path)
print("結果の保存先:", experiment.output_dir)
for item in experiment.recordings:
    ratio = item.get("third_played_ratio", "-")
    print(f"  {item['id']:24} 元素材 {item['source_id']:24} {item['split']:10} {item['material']:9} 3度の割合 {ratio}")
print("伴奏:", [item["id"] for item in experiment.accompaniments])

## 4. 件数と分割を確かめる

In [ ]:
problems = check_source_split(experiment.recordings)
print("元素材単位の分割:", "問題なし" if not problems else problems)
counts = count_materials(experiment, experiment.recordings)
print(f"元録音数 {counts['source_recordings']}、切り出し数 {counts['clips']}、"
      f"人工混合の条件数 {counts['artificial_mixture_conditions']}（伴奏 {counts['accompaniments']} 種 × 音量 {len(LEVELS_DB)} 段階）")

## 5. 実験を実行する

`RUN_SEPARATION = False` にすると、Demucs を使わずに solo と mixture だけを実行します（分離の効果は調べられません）。

評価用の録音だけを使うときは `SPLIT = "evaluation"`、すべて使うときは `"all"` にします。

In [ ]:
RUN_SEPARATION = True
SPLIT = "evaluation"

conditions = ["solo", "mixture", "separated"] if RUN_SEPARATION else ["solo", "mixture"]
result = run_experiment(experiment, split=SPLIT, conditions=conditions)

## 6. 結果を見る

保存したファイル：

| ファイル | 内容 |
| --- | --- |
| `items.csv` | 1行＝1つの録音・条件・伴奏の音量の評価結果（処理時間、時間ずれの確認値を含む） |
| `paired_differences.csv` | 同じ録音・伴奏・音量の「分離後 − 分離前」の差 |
| `summary_by_condition.csv` | 素材・条件・伴奏・音量ごとの正解率の平均、標準偏差、最小、最大 |
| `summary_differences.csv` | 素材・伴奏・音量ごとの「分離後 − 分離前」の平均、ばらつき |
| `experiment.json` | 設定ファイルの内容、実際に使った設定（省略した項目の既定値を含む）、件数、ライブラリの版、コミット |
| `summary.png` | 条件別の正解率、分離前後の差、処理時間の図 |
| `items/` | 条件ごとの推定結果（chords.csv、analysis.json）と音（WAV） |

- 集計では、同じ元素材から複数の切り出しがあるとき、まず元素材ごとに平均してから、元素材の間で平均やばらつきを計算します。`n_sources` は元素材の数、`n_clips` は切り出しの数です。
- 集計のファイルは実行のたびに書き直しますが、`items/` には前の実行のファイルが残ることがあります。条件を変えて比べるときは、`run_experiment(experiment, split=SPLIT, conditions=conditions, output_dir="outputs/experiment_level18")` のように保存先を分けましょう。

In [ ]:
display(Image(filename=str(experiment.output_dir / "summary.png")))

In [ ]:
print("録音ごとの「分離後 − 分離前」")
print(f"{'録音':24} {'伴奏[dB]':>8} {'solo':>6} {'mixture':>8} {'separated':>10} {'差':>7}")
for pair in result["pairs"]:
    print(f"{pair['recording_id']:24} {pair['accompaniment_db']:>8} {pair['solo_accuracy']:>6} "
          f"{pair['mixture_accuracy']:>8} {pair['separated_accuracy']:>10} {pair['difference']:>+7.3f}")

In [ ]:
print("時間軸の確認（0 ms なら単独ギターと時刻がそろっています）")
for row in result["rows"]:
    if "lag_vs_solo_ms" in row:
        print(f"  {row['recording_id']:24} {row['condition']:9} {str(row['accompaniment_db']):>5} dB  ずれ {row['lag_vs_solo_ms']} ms")

## 7. 1つの録音を聴いて、図で確かめる

平均の値だけでなく、個々の録音を聴いて確かめましょう。特に、分離後に正解率が下がった録音は、何が起きたのかを耳と図で調べます。

- `RECORDING_ID`、`ACCOMPANIMENT_ID`、`LEVEL_DB` を変えると、別の録音・伴奏・音量を選べます。
- 単独ギター、混合音、分離したギター（今回の実行で保存したWAV）を再生します。再生の音量は自動でそろえないので、音の大きさの違いもそのまま聞こえます。
- 使うのは、**今回の実行結果（`result`）にある条件だけ**です。`items/` には前の実行のファイルが残ることがありますが、今回の実行で保存し、内容と長さが今回の解析と一致するファイルだけを使います。今回の実行にない条件（例：分離しなかったときの separated）や、音を保存していない条件（`save_audio = false`）は、理由を表示して省きます。
- 聴ける音を実験と同じ設定で解析し直し、コード推移を並べます。続けて、1つの条件のクロマグラムを表示します。解析し直した結果は、`items.csv` の値とわずかに違うことがあります（分離した音を保存・読み込みし直すため）。

In [ ]:
RECORDING_ID = result["rows"][0]["recording_id"]  # 聴きたい録音（例: "01_Rock3-117-Bb_comp"）
ACCOMPANIMENT_ID = experiment.accompaniments[0]["id"] if experiment.accompaniments else None  # 聴きたい伴奏
LEVEL_DB = LEVELS_DB[-1]                          # 聴きたい伴奏の音量 [dB]（合奏録音では使いません）

# 今回の実行結果（result）にある条件の音だけを読み込みます（前の実行で残ったファイルは使いません）
listening = load_listening_clips(experiment, result, RECORDING_ID, ACCOMPANIMENT_ID, LEVEL_DB)
for note in listening.notes:
    print("聴けない条件 →", note)
for name, clip in listening.clips.items():
    print(f"{name}（{clip.duration_s:.2f} 秒）")
    display(Audio(np.clip(clip.samples, -1.0, 1.0), rate=clip.sample_rate, normalize=False))

In [ ]:
analyses = {name: recognise_chords(clip, experiment.settings) for name, clip in listening.clips.items()}
if analyses:
    plot_comparison(analyses, listening.reference, f"{RECORDING_ID}, {listening.description}")
    plt.show()
    CONDITION = next(name for name in ["separated", "mixture", "solo"] if name in analyses)  # クロマグラムを見る条件
    plot_analysis(analyses[CONDITION], f"{RECORDING_ID} / {CONDITION}", listening.reference)
    plt.show()
else:
    print("今回の実行結果には、聴き比べられる条件がありません（上の理由を確かめてください）")

## 8. 結果を解釈するときの注意

- この例は**5録音・15混合条件**の小さな実験です。平均の差は録音によるばらつきに比べて小さいことがあり、一般的な結論にはなりません。
- 伴奏は**プログラムで作ったドラム**で、和音を持ちません。ベースやキーボードなど、和音の音を鳴らす伴奏では結果が変わる可能性があります。人工的な混合音の結果を、実際の合奏録音の結果として述べないでください。
- 正解は「指示されたコード（リードシート）」です。演奏者が実際に弾いた音とは一致しない区間があります。
- 結果は、使った音源・分離モデル（htdemucs_6s）・コード認識器・設定の範囲でのものです。
- 「なぜそうなったか」の推測（例：ドラムの雑音がクロマを平らにする）と、実験で確認できた事実（例：正解率の差）を分けて書きましょう。

## 9. 練習

1. `LEVELS_DB` に 18.0 を加えて、伴奏がさらに大きいときの傾向を調べましょう。
2. `EXAMPLE_TRACKS` 以外の録音IDを `prepare_guitarset_experiment` に渡して、素材を増やしましょう。`third_played_ratio` も確認します。
3. 設定（例：`segment_seconds`）を変えて比べたいときは、調整用（tuning）の録音だけで設定を選び、選んだ設定で評価用の録音を1回だけ評価します。既定の素材には調整用の録音がない（`SPLIT = "tuning"` にするとエラーになる）ので、調整用の録音を加える方法も考えましょう。
4. 自分の録音を加えるときは、`docs/research-guide.md` の手順に従い、正解注釈CSV（start_s, end_s, chord）を作りましょう。

### 解答例・考え方

1. 伴奏が大きいほど混合音の正解率は下がりやすく、分離の効果（差）が大きくなる傾向があります。ただし、分離後の結果も単独ギターの結果より良くなるとは限りません。
2. 素材を増やすと、平均値の信頼性が上がり、スタイルや奏者による違いも調べられます。元素材の数と混合条件の数を分けて報告します。
3. 評価用の結果を見ながら設定を選ぶと、評価用の録音に合わせすぎた（過大な）結果になります。調整用で決めた設定を固定し、評価用は最後に1回だけ使います。奏者00の録音を加えると、奏者00が調整用になります（既定の設定を選んだときと同じ分け方です）。設定を変えるには `replace` を使います。

```python
from dataclasses import replace

config_path = prepare_guitarset_experiment(
    EXAMPLE_TRACKS + ["00_Rock3-117-Bb_comp"], "data/raw/guitarset",
    "data/raw/guitarset/experiment_tuning.toml", duration_s=20.0, levels_db=LEVELS_DB,
)
experiment = load_experiment(config_path)
experiment.settings = replace(experiment.settings, segment_seconds=1.0)  # 試す設定
result = run_experiment(experiment, split="tuning", conditions=conditions, output_dir="outputs/experiment_tuning")
```
4. 注釈の時刻は、注釈したファイルの先頭を0秒とします。切り出して使う場合は、設定ファイルの `start_s` と `duration_s` で指定すれば、注釈の時刻は自動で切り出し範囲に合わせられます。